# GeoMarketing IDF — J4 : accessibilité en transports

**Objectif :** enrichir le profil socio-économique du J3 avec des indicateurs communaux de desserte en transports collectifs.

Ce notebook :

1. télécharge le jeu **Arrêts et lignes associées** d'Île-de-France Mobilités ;
2. conserve les communes franciliennes à partir de leur code Insee ;
3. distingue bus, métro, tramway, RER, Transilien et autres modes ;
4. compte les points d'arrêt et les lignes par commune ;
5. joint ces résultats au fichier du J3 sans supprimer ses colonnes ;
6. exporte une nouvelle table complète.

Source officielle : [Île-de-France Mobilités — Arrêts et lignes associées](https://data.iledefrance-mobilites.fr/explore/dataset/arrets-lignes/).

> Attention : pour le bus, deux points situés de part et d'autre d'une rue peuvent être deux identifiants différents. Les nombres décrivent donc des **points d'arrêt de référence**, pas toujours des stations physiques uniques.

## 1. Préparer Python

Si une ancienne tentative d'import a échoué, utilise **Kernel → Restart Kernel**, puis relance toutes les cellules.

In [1]:
import os

for variable in ("OMP_NUM_THREADS", "NUMEXPR_NUM_THREADS", "MKL_NUM_THREADS"):
    valeur = os.environ.get(variable)
    if valeur is not None:
        try:
            int(valeur)
        except ValueError:
            print(f"Variable invalide supprimée : {variable}={valeur!r}")
            os.environ.pop(variable, None)

from pathlib import Path
from urllib.request import Request, urlopen
import shutil

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 100)
print("Environnement Python prêt.")

Variable invalide supprimée : OMP_NUM_THREADS='A'
Environnement Python prêt.


## 2. Retrouver le projet et le fichier du J3

Modifie uniquement `DOSSIER_PROJET_MANUEL` si la détection automatique échoue.

In [2]:
DOSSIER_PROJET_MANUEL = None
# Exemple : Path(r"C:\Users\VotreNom\OneDrive\GeoMarketing_IDF")

def trouver_projet():
    if DOSSIER_PROJET_MANUEL is not None:
        return Path(DOSSIER_PROJET_MANUEL)

    candidats = []
    for variable in ("OneDrive", "OneDriveConsumer", "OneDriveCommercial"):
        racine = os.environ.get(variable)
        if racine:
            candidats.extend([
                Path(racine) / "GeoMarketing_IDF",
                Path(racine) / "Documents" / "GeoMarketing_IDF",
            ])

    candidats.extend([
        Path.home() / "OneDrive" / "GeoMarketing_IDF",
        Path.home() / "Documents" / "GeoMarketing_IDF",
        Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd(),
    ])

    for candidat in candidats:
        if (candidat / "data").exists():
            return candidat.resolve()

    raise FileNotFoundError(
        "Dossier GeoMarketing_IDF introuvable. Renseigne DOSSIER_PROJET_MANUEL."
    )

PROJET = trouver_projet()
RAW = PROJET / "data" / "raw" / "idfm"
PROCESSED = PROJET / "data" / "processed" / "insee"
RAW.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

FICHIER_J3 = PROCESSED / "profil_socioeconomique_idf.csv"
FICHIER_SORTIE = PROCESSED / "profil_geomarketing_idf_j4.csv"
FICHIER_ARRETS = RAW / "arrets_lignes_idfm.csv"

print("Projet :", PROJET)
print("Entrée J3 :", FICHIER_J3)
print("Sortie J4 :", FICHIER_SORTIE)
assert FICHIER_J3.exists(), f'Fichier J3 absent : {FICHIER_J3}'

Projet : C:\Users\almou\OneDrive\GeoMarketing_IDF
Entrée J3 : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\insee\profil_socioeconomique_idf.csv
Sortie J4 : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\insee\profil_geomarketing_idf_j4.csv


## 3. Télécharger les arrêts et les lignes

Le fichier est conservé dans `data/raw/idfm`. Mets `FORCER_TELECHARGEMENT = True` uniquement si tu veux actualiser ultérieurement les données.

In [3]:
URL_ARRETS = (
    "https://data.iledefrance-mobilites.fr/api/explore/v2.1/catalog/"
    "datasets/arrets-lignes/exports/csv?lang=fr&timezone=Europe%2FParis"
    "&use_labels=false&delimiter=%3B"
)
FORCER_TELECHARGEMENT = False

if FORCER_TELECHARGEMENT or not FICHIER_ARRETS.exists():
    print("Téléchargement des données IDFM (environ 14 Mo)...")
    fichier_temporaire = FICHIER_ARRETS.with_suffix(".tmp")
    requete = Request(URL_ARRETS, headers={"User-Agent": "Mozilla/5.0"})
    try:
        with urlopen(requete, timeout=180) as reponse, open(fichier_temporaire, "wb") as sortie:
            shutil.copyfileobj(reponse, sortie)
        assert fichier_temporaire.stat().st_size > 5_000_000, "Téléchargement incomplet."
        fichier_temporaire.replace(FICHIER_ARRETS)
    finally:
        if fichier_temporaire.exists():
            fichier_temporaire.unlink()
    print("Téléchargement terminé.")
else:
    print("Fichier déjà présent :", FICHIER_ARRETS)

arrets = pd.read_csv(
    FICHIER_ARRETS,
    sep=";",
    dtype=str,
    encoding="utf-8-sig",
    low_memory=False,
)
arrets.columns = arrets.columns.str.strip().str.lower()
print(f"{len(arrets):,} relations arrêt-ligne chargées.")
display(arrets.head())

Téléchargement des données IDFM (environ 14 Mo)...
Téléchargement terminé.
74,959 relations arrêt-ligne chargées.


,id,route_long_name,stop_id,stop_name,stop_lon,stop_lat,operatorname,shortname,bookingrules,mode,pointgeo,nom_commune,code_insee
0,IDFM:C01389,T1,IDFM:23293,Chemin des Reniers,2.321527043611886,48.934560806805955,RATP,T1,NaN,Tramway,"48.934560806805955, 2.321527043611886",Villeneuve-la-Garenne,92078
1,IDFM:C01389,T1,IDFM:23313,Théâtre Gérard Philipe,2.350468785750953,48.93749994624904,RATP,T1,NaN,Tramway,"48.93749994624904, 2.350468785750953",Saint-Denis,93066
2,IDFM:C01389,T1,IDFM:22268,Marché de Saint-Denis,2.35587486964062,48.93856329762427,RATP,T1,NaN,Tramway,"48.93856329762427, 2.35587486964062",Saint-Denis,93066
3,IDFM:C01389,T1,IDFM:24429,Stade Géo André,2.4020687684654924,48.92433314315498,RATP,T1,NaN,Tramway,"48.92433314315498, 2.4020687684654924",La Courneuve,93027
4,IDFM:C01389,T1,IDFM:24427,Danton,2.406618691605552,48.922656100102074,RATP,T1,NaN,Tramway,"48.922656100102074, 2.406618691605552",La Courneuve,93027


## 4. Nettoyer les codes communaux et les modes

In [5]:
colonnes_requises = {
    "id", "stop_id", "stop_name", "shortname", "mode", "nom_commune", "code_insee"
}
manquantes = colonnes_requises - set(arrets.columns)
assert not manquantes, f"Colonnes IDFM manquantes : {sorted(manquantes)}"

idf_departements = {"75", "77", "78", "91", "92", "93", "94", "95"}
arrets["CODGEO"] = arrets["code_insee"].astype("string").str.extract(r"(\d{5})$")[0]
arrets = arrets.loc[arrets["CODGEO"].str[:2].isin(idf_departements)].copy()

correspondance_modes = {
    "Bus": "BUS",
    "Metro": "METRO",
    "Tramway": "TRAMWAY",
    "RapidTransit": "RER",
    "LocalTrain": "TRANSILIEN",
    "regionalRail": "TER",
    "RailShuttle": "NAVETTE_FERROVIAIRE",
    "CableWay": "TELEPHERIQUE",
    "Funicular": "FUNICULAIRE",
}
arrets["MODE_GROUPE"] = arrets["mode"].map(correspondance_modes).fillna("AUTRE")

for col in ("id", "stop_id", "shortname", "stop_name"):
    arrets[col] = arrets[col].astype("string").str.strip()

arrets = arrets.dropna(subset=["CODGEO", "id", "stop_id"])
arrets = arrets.drop_duplicates(["CODGEO", "id", "stop_id"])

print("Communes présentes dans les données IDFM :", arrets["CODGEO"].nunique())
display(arrets["MODE_GROUPE"].value_counts().rename("RELATIONS_ARRET_LIGNE"))
assert arrets["CODGEO"].str.fullmatch(r"\d{5}").all()

Communes présentes dans les données IDFM : 1261


MODE_GROUPE
BUS                    72712
METRO                    803
TRAMWAY                  564
RER                      248
TRANSILIEN               235
TER                       82
NAVETTE_FERROVIAIRE       16
TELEPHERIQUE              10
FUNICULAIRE                4
Name: RELATIONS_ARRET_LIGNE, dtype: int64

## 5. Calculer les indicateurs communaux

Les comptes utilisent des identifiants distincts afin d'éviter de compter plusieurs fois la même relation arrêt-ligne.

In [6]:
def liste_unique(serie):
    valeurs = sorted({str(x).strip() for x in serie.dropna() if str(x).strip()})
    return " | ".join(valeurs)

transport = (
    arrets.groupby("CODGEO", as_index=False)
    .agg(
        NOM_COMMUNE_IDFM=("nom_commune", "first"),
        POINTS_ARRET_TOTAL=("stop_id", "nunique"),
        LIGNES_TOTAL=("id", "nunique"),
        NB_MODES_TRANSPORT=("MODE_GROUPE", "nunique"),
        MODES_PRESENTS=("MODE_GROUPE", liste_unique),
    )
)

modes_a_mesurer = ["BUS", "METRO", "TRAMWAY", "RER", "TRANSILIEN", "TER"]

for mode in modes_a_mesurer:
    sous_table = arrets.loc[arrets["MODE_GROUPE"].eq(mode)]
    indicateurs_mode = (
        sous_table.groupby("CODGEO", as_index=False)
        .agg(**{
            f"POINTS_ARRET_{mode}": ("stop_id", "nunique"),
            f"LIGNES_{mode}": ("id", "nunique"),
            f"LISTE_LIGNES_{mode}": ("shortname", liste_unique),
        })
    )
    transport = transport.merge(indicateurs_mode, on="CODGEO", how="left", validate="one_to_one")

modes_lourds = {"METRO", "TRAMWAY", "RER", "TRANSILIEN"}
sous_table_lourde = arrets.loc[arrets["MODE_GROUPE"].isin(modes_lourds)]
lourd = (
    sous_table_lourde.groupby("CODGEO", as_index=False)
    .agg(
        POINTS_ARRET_TRANSPORT_LOURD=("stop_id", "nunique"),
        LIGNES_TRANSPORT_LOURD=("id", "nunique"),
        NB_MODES_LOURDS=("MODE_GROUPE", "nunique"),
    )
)
transport = transport.merge(lourd, on="CODGEO", how="left", validate="one_to_one")

colonnes_compte = [
    c for c in transport.columns
    if c.startswith("POINTS_ARRET_") or c.startswith("LIGNES_") or c in ["NB_MODES_TRANSPORT", "NB_MODES_LOURDS"]
]
colonnes_compte = [c for c in colonnes_compte if not c.startswith("LISTE_")]
transport[colonnes_compte] = transport[colonnes_compte].fillna(0).astype("int64")

colonnes_liste = [c for c in transport.columns if c.startswith("LISTE_LIGNES_")]
transport[colonnes_liste] = transport[colonnes_liste].fillna("")

assert transport["CODGEO"].is_unique
print("Dimensions de la table transport :", transport.shape)
display(transport.head())

Dimensions de la table transport : (1261, 27)


,CODGEO,NOM_COMMUNE_IDFM,POINTS_ARRET_TOTAL,LIGNES_TOTAL,NB_MODES_TRANSPORT,MODES_PRESENTS,POINTS_ARRET_BUS,LIGNES_BUS,LISTE_LIGNES_BUS,POINTS_ARRET_METRO,LIGNES_METRO,LISTE_LIGNES_METRO,POINTS_ARRET_TRAMWAY,LIGNES_TRAMWAY,LISTE_LIGNES_TRAMWAY,POINTS_ARRET_RER,LIGNES_RER,LISTE_LIGNES_RER,POINTS_ARRET_TRANSILIEN,LIGNES_TRANSILIEN,LISTE_LIGNES_TRANSILIEN,POINTS_ARRET_TER,LIGNES_TER,LISTE_LIGNES_TER,POINTS_ARRET_TRANSPORT_LOURD,LIGNES_TRANSPORT_LOURD,NB_MODES_LOURDS
0,75056,Paris,3436,251,7,BUS | FUNICULAIRE | METRO | RER | TER | TRAMWA...,2622,208,102 | 105 | 109 | 111 | 112 | 114 | 115 | 118 ...,648,16,1 | 10 | 11 | 12 | 13 | 14 | 2 | 3 | 3B | 4 | ...,124,4,T2 | T3a | T3b | T9,29,5,A | B | C | D | E,6,7,H | J | K | L | N | P | R,11,10,TER,805,32,4
1,77001,Achères-la-Forêt,7,2,1,BUS,7,2,3453 | 3459,0,0,,0,0,,0,0,,0,0,,0,0,,0,0,0
2,77002,Amillis,7,1,1,BUS,7,1,2460,0,0,,0,0,,0,0,,0,0,,0,0,,0,0,0
3,77003,Amponville,4,1,1,BUS,4,1,3556,0,0,,0,0,,0,0,,0,0,,0,0,,0,0,0
4,77004,Andrezel,4,4,1,BUS,4,4,2477 | 3120 | 3132 | 3135,0,0,,0,0,,0,0,,0,0,,0,0,,0,0,0


## 6. Joindre les transports au fichier complet du J3

La jointure part de `profil_j3` : toutes les colonnes démographiques et socio-économiques sont donc conservées.

In [7]:
def lire_csv_robuste(chemin):
    for encodage in ("utf-8-sig", "utf-8", "cp1252"):
        for sep in (";", ","):
            try:
                df = pd.read_csv(chemin, sep=sep, dtype=str, encoding=encodage)
                if len(df.columns) > 1:
                    return df
            except (UnicodeDecodeError, pd.errors.ParserError):
                continue
    raise ValueError(f"Impossible de lire {chemin}")

profil_j3 = lire_csv_robuste(FICHIER_J3)
profil_j3.columns = profil_j3.columns.str.strip()
assert "CODGEO" in profil_j3.columns, "CODGEO absent du fichier J3."
profil_j3["CODGEO"] = profil_j3["CODGEO"].astype("string").str.strip().str.zfill(5)
assert profil_j3["CODGEO"].is_unique, "CODGEO n'est pas unique dans le fichier J3."

codes_hors_profil = sorted(set(transport["CODGEO"]) - set(profil_j3["CODGEO"]))
print("Codes transport absents du J3 :", len(codes_hors_profil))
if codes_hors_profil:
    print(codes_hors_profil[:20])

profil_j4 = profil_j3.merge(
    transport,
    on="CODGEO",
    how="left",
    validate="one_to_one",
    indicator=True,
)

print("Communes du J3 :", len(profil_j3))
print("Communes du J4 :", len(profil_j4))
print("Colonnes du J3 :", len(profil_j3.columns))
print("Colonnes du J4 :", len(profil_j4.columns) - 1)
assert len(profil_j4) == len(profil_j3), "Le nombre de communes a changé pendant la jointure."

sans_arret = profil_j4["_merge"].eq("left_only").sum()
print("Communes sans point d'arrêt recensé :", sans_arret)
profil_j4 = profil_j4.drop(columns="_merge")

Codes transport absents du J3 : 0
Communes du J3 : 1266
Communes du J4 : 1266
Colonnes du J3 : 34
Colonnes du J4 : 60
Communes sans point d'arrêt recensé : 5


## 7. Finaliser les indicateurs et créer une catégorie descriptive

In [8]:
colonnes_transport_numeriques = [
    c for c in transport.columns
    if c.startswith("POINTS_ARRET_") or c.startswith("LIGNES_") or c in ["NB_MODES_TRANSPORT", "NB_MODES_LOURDS"]
]
colonnes_transport_numeriques = [c for c in colonnes_transport_numeriques if not c.startswith("LISTE_")]

for col in colonnes_transport_numeriques:
    profil_j4[col] = pd.to_numeric(profil_j4[col], errors="coerce").fillna(0).astype("int64")

colonnes_transport_texte = ["NOM_COMMUNE_IDFM", "MODES_PRESENTS"] + [
    c for c in transport.columns if c.startswith("LISTE_LIGNES_")
]
for col in colonnes_transport_texte:
    profil_j4[col] = profil_j4[col].fillna("")

def trouver_population(df):
    for col in ("POP_2022", "P22_POP", "POPULATION_2022"):
        if col in df.columns:
            return col
    return None

col_population = trouver_population(profil_j4)
if col_population:
    profil_j4[col_population] = pd.to_numeric(
        profil_j4[col_population].astype("string").str.replace(" ", "", regex=False).str.replace(",", ".", regex=False),
        errors="coerce",
    )
    profil_j4["POINTS_ARRET_POUR_10000_HAB"] = (
        10_000 * profil_j4["POINTS_ARRET_TOTAL"] / profil_j4[col_population]
    ).round(2)
    profil_j4["POINTS_ARRET_LOURD_POUR_100000_HAB"] = (
        100_000 * profil_j4["POINTS_ARRET_TRANSPORT_LOURD"] / profil_j4[col_population]
    ).round(2)

conditions = [
    profil_j4["POINTS_ARRET_TOTAL"].eq(0),
    profil_j4["NB_MODES_LOURDS"].eq(0),
    profil_j4["NB_MODES_LOURDS"].eq(1),
]
choix = ["AUCUN_ARRET", "BUS_OU_AUTRE_UNIQUEMENT", "UN_MODE_LOURD"]
profil_j4["CATEGORIE_DESSERTE"] = np.select(
    conditions, choix, default="PLUSIEURS_MODES_LOURDS"
)

display(profil_j4["CATEGORIE_DESSERTE"].value_counts().rename("NB_COMMUNES"))

CATEGORIE_DESSERTE
BUS_OU_AUTRE_UNIQUEMENT    929
UN_MODE_LOURD              258
PLUSIEURS_MODES_LOURDS      74
AUCUN_ARRET                  5
Name: NB_COMMUNES, dtype: int64

## 8. Contrôler quelques communes

Le tableau permet notamment de comparer Saint-Denis, Argenteuil, Villiers-le-Bel et Garges-lès-Gonesse.

In [9]:
assert "93066" in set(profil_j4["CODGEO"]), "Saint-Denis (93066) est absent."
assert "93059" not in set(profil_j4["CODGEO"]), "Pierrefitte-sur-Seine apparaît encore séparément."

communes_test = ["93066", "95018", "95680", "95268", "95585"]
colonnes_affichage = [
    c for c in [
        "CODGEO", "LIBGEO", "NOM_COM", "NOM_COMMUNE_IDFM",
        "POINTS_ARRET_TOTAL", "LIGNES_TOTAL", "POINTS_ARRET_BUS", "LIGNES_BUS",
        "POINTS_ARRET_METRO", "LISTE_LIGNES_METRO",
        "POINTS_ARRET_TRAMWAY", "LISTE_LIGNES_TRAMWAY",
        "POINTS_ARRET_RER", "LISTE_LIGNES_RER",
        "POINTS_ARRET_TRANSILIEN", "LISTE_LIGNES_TRANSILIEN",
        "NB_MODES_LOURDS", "CATEGORIE_DESSERTE"
    ] if c in profil_j4.columns
]
display(profil_j4.loc[profil_j4["CODGEO"].isin(communes_test), colonnes_affichage])
print("Contrôle de Saint-Denis et des communes tests réussi.")

,CODGEO,NOM_COMMUNE_IDFM,POINTS_ARRET_TOTAL,LIGNES_TOTAL,POINTS_ARRET_BUS,LIGNES_BUS,POINTS_ARRET_METRO,LISTE_LIGNES_METRO,POINTS_ARRET_TRAMWAY,LISTE_LIGNES_TRAMWAY,POINTS_ARRET_RER,LISTE_LIGNES_RER,POINTS_ARRET_TRANSILIEN,LISTE_LIGNES_TRANSILIEN,NB_MODES_LOURDS,CATEGORIE_DESSERTE
1027,93066,Saint-Denis,311,41,248,31,12,12 | 13 | 14,47,T1 | T11 | T5 | T8,4,B | D,1,H,4,PLUSIEURS_MODES_LOURDS
1088,95018,Argenteuil,262,22,260,21,0,,0,,0,,2,J,1,UN_MODE_LOURD
1157,95268,Garges-lès-Gonesse,75,14,74,13,0,,0,,1,D,0,,1,UN_MODE_LOURD
1240,95585,Sarcelles,128,21,118,20,0,,10,T5,0,,0,,1,UN_MODE_LOURD
1263,95680,Villiers-le-Bel,46,6,46,6,0,,0,,0,,0,,0,BUS_OU_AUTRE_UNIQUEMENT


Contrôle de Saint-Denis et des communes tests réussi.


## 9. Contrôles qualité et export

In [10]:
assert profil_j4["CODGEO"].is_unique
assert profil_j4["CODGEO"].str.fullmatch(r"\d{5}").all()
assert len(profil_j4) == len(profil_j3)

for col in colonnes_transport_numeriques:
    assert (profil_j4[col] >= 0).all(), f"Valeur négative dans {col}."

print("Nombre de colonnes conservées depuis le J3 :", len(profil_j3.columns))
print("Nombre total de colonnes du J4 :", len(profil_j4.columns))

profil_j4.to_csv(FICHIER_SORTIE, index=False, encoding="utf-8-sig")
print("Fichier créé :", FICHIER_SORTIE)
print("Nombre de communes :", len(profil_j4))
print("J4 terminé ✅")

Nombre de colonnes conservées depuis le J3 : 34
Nombre total de colonnes du J4 : 63
Fichier créé : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\insee\profil_geomarketing_idf_j4.csv
Nombre de communes : 1266
J4 terminé ✅


## Résultat du J4

Le fichier suivant est maintenant disponible :

`data/processed/insee/profil_geomarketing_idf_j4.csv`

Il conserve les données démographiques et socio-économiques précédentes, auxquelles s'ajoutent les indicateurs de desserte en transports.

Enregistre ce notebook dans le dépôt GitHub sous :

`notebooks/04_accessibilite_transports_idf.ipynb`

Résumé du commit : `Ajout des indicateurs de transports IDF`